In [2]:
import os

print(os.getcwd())

c:\Users\BIT\OneDrive\Desktop\ONGC_Complaint_Analytics\notebooks


In [3]:
print(os.getcwd())

c:\Users\BIT\OneDrive\Desktop\ONGC_Complaint_Analytics\notebooks


In [6]:
import os

os.listdir()

['EDA.ipynb', 'Phase2_NewDataset.ipynb', 'Phase4_ML.ipynb']

In [7]:
df = pd.read_excel(
    "../data/raw/cleaned_training_sheet ONGC.xlsx"
)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Ticket_No                2000 non-null   str           
 1   Title                    1802 non-null   str           
 2   Description              2000 non-null   str           
 3   Category                 2000 non-null   str           
 4   HW_Flag                  2000 non-null   int64         
 5   Priority                 2000 non-null   str           
 6   Department               2000 non-null   str           
 7   Status                   2000 non-null   str           
 8   Resolution_Time_Hours    1693 non-null   float64       
 9   Date_created             2000 non-null   datetime64[us]
 10  date_assigned            2000 non-null   datetime64[us]
 11  Assigned_To              2000 non-null   str           
 12  Closing_Date             2000 non-null   date

In [9]:
df.isnull().sum()

Ticket_No                    0
Title                      198
Description                  0
Category                     0
HW_Flag                      0
Priority                     0
Department                   0
Status                       0
Resolution_Time_Hours      307
Date_created                 0
date_assigned                0
Assigned_To                  0
Closing_Date                 0
Sentiment_Label              0
Interpretation_Software      0
System_hostname              0
Group                        0
dtype: int64

In [10]:
df["Assigned_To"] = (
    df["Assigned_To"]
    .fillna("Unknown")
)

In [11]:
df["Date_created"] = pd.to_datetime(
    df["Date_created"]
)

df["Closing_Date"] = pd.to_datetime(
    df["Closing_Date"]
)

In [12]:
df["Ticket_Age_Days"] = (
    df["Closing_Date"]
    - df["Date_created"]
).dt.days

In [13]:
df["Month"] = (
    df["Date_created"]
    .dt.month
)

In [14]:
df["Weekday"] = (
    df["Date_created"]
    .dt.day_name()
)

In [15]:
df["Hardware_Issue"] = df["HW_Flag"]

In [16]:
df.to_excel(
    "../data/cleaned/cleaned_complaints_v2.xlsx",
    index=False
)

In [17]:
df.groupby("Month").size()

Month
1     160
2     142
3     158
4     177
5     181
6     171
7     158
8     170
9     160
10    174
11    164
12    185
dtype: int64

In [18]:
df["Department"].value_counts()

Department
Technical Support                  462
Product Support                    293
IT Support                         282
Customer Service                   214
Billing and Payments               138
Applications                       110
Security                           100
Infrastructure                      99
Data Center                         84
Returns and Exchanges               70
Service Outages and Maintenance     50
Sales and Pre-Sales                 43
Human Resources                     34
General Inquiry                     21
Name: count, dtype: int64

In [19]:
df["Assigned_To"].value_counts()

Assigned_To
Karan Joshi     223
Pooja Desai     222
Amit Sharma     219
Sneha Nair      215
Rahul Verma     196
Priya Singh     191
Vikas Kumar     189
Neha Gupta      185
Anjali Patel    184
Rohit Mehta     176
Name: count, dtype: int64

In [20]:
df["Group"].value_counts()

Group
KS    554
MO    510
KK    493
TD    443
Name: count, dtype: int64

In [21]:
df["Interpretation_Software"].value_counts()

Interpretation_Software
PaleoScan        418
Petrel           414
OpenWorks        400
GeoGraphix       387
Kingdom Suite    381
Name: count, dtype: int64

In [22]:
df["HW_Flag"].value_counts()

HW_Flag
0    1927
1      73
Name: count, dtype: int64

In [23]:
df.groupby(
    "Department"
)["Resolution_Time_Hours"].mean()

Department
Applications                       124.097826
Billing and Payments                83.837838
Customer Service                    89.405405
Data Center                        121.179104
General Inquiry                     82.473684
Human Resources                     96.266667
IT Support                          98.460000
Infrastructure                     112.272727
Product Support                     81.262295
Returns and Exchanges               88.603448
Sales and Pre-Sales                 93.225000
Security                           118.765432
Service Outages and Maintenance     95.534884
Technical Support                   88.683117
Name: Resolution_Time_Hours, dtype: float64

In [24]:
df["combined_text"] = (

    df["Title"].astype(str)

    + " "

    + df["Description"].astype(str)

    + " "

    + df["Department"].astype(str)

    + " "

    + df["Group"].astype(str)

    + " "

    + df["Interpretation_Software"].astype(str)

)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

In [26]:
y = df["Priority"]

In [27]:
X = df["combined_text"]

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [33]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

In [35]:
df["combined_text"].isnull().sum()

np.int64(198)

In [37]:
df["combined_text"] = (
    df["combined_text"]
    .fillna("")
)

In [38]:
df["combined_text"].isnull().sum()

np.int64(0)

In [39]:
X_tfidf = tfidf.fit_transform(
    df["combined_text"]
)

In [40]:
from sklearn.model_selection import train_test_split

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [42]:
from sklearn.ensemble import RandomForestClassifier

In [43]:
priority_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

In [44]:
priority_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [45]:
preds = priority_model.predict(
    X_test
)

In [46]:
from sklearn.metrics import accuracy_score

In [47]:
accuracy = accuracy_score(
    y_test,
    preds
)

In [48]:
print(
    "Accuracy:",
    accuracy
)

Accuracy: 0.4475


In [49]:
from sklearn.metrics import classification_report

In [50]:
print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

    Critical       0.23      0.40      0.29        25
        High       0.18      0.10      0.13        30
         Low       0.29      0.25      0.27        28
      Medium       0.31      0.24      0.27        21
        high       0.56      0.63      0.59       123
         low       0.60      0.05      0.09        59
      medium       0.47      0.65      0.55       114

    accuracy                           0.45       400
   macro avg       0.38      0.33      0.31       400
weighted avg       0.46      0.45      0.41       400



In [51]:
import joblib

In [52]:
joblib.dump(
    priority_model,
    "../models/priority_model.pkl"
)

['../models/priority_model.pkl']

In [53]:
joblib.dump(
    tfidf,
    "../models/tfidf.pkl"
)

['../models/tfidf.pkl']

In [55]:
print(df.columns.tolist())

['Ticket_No', 'Title', 'Description', 'Category', 'HW_Flag', 'Priority', 'Department', 'Status', 'Resolution_Time_Hours', 'Date_created', 'date_assigned', 'Assigned_To', 'Closing_Date', 'Sentiment_Label', 'Interpretation_Software', 'System_hostname', 'Group', 'Ticket_Age_Days', 'Month', 'Weekday', 'Hardware_Issue', 'combined_text']


In [56]:
import numpy as np

df["SLA_Status"] = np.where(
    df["Resolution_Time_Hours"] > 48,
    "SLA Breached",
    "Within SLA"
)

In [57]:
df["SLA_Status"].value_counts()

SLA_Status
SLA Breached    1275
Within SLA       725
Name: count, dtype: int64

In [58]:
X = X_tfidf

y = df["SLA_Status"]

In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [60]:
from sklearn.ensemble import RandomForestClassifier

sla_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    random_state=42,
    class_weight="balanced"
)

In [61]:
sla_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 

In [62]:
preds = sla_model.predict(
    X_test
)

In [63]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

print(
    accuracy_score(
        y_test,
        preds
    )
)

print(
    classification_report(
        y_test,
        preds
    )
)

0.5575
              precision    recall  f1-score   support

SLA Breached       0.64      0.68      0.66       255
  Within SLA       0.38      0.34      0.36       145

    accuracy                           0.56       400
   macro avg       0.51      0.51      0.51       400
weighted avg       0.55      0.56      0.55       400



In [64]:
import joblib

joblib.dump(
    sla_model,
    "../models/sla_model.pkl"
)

['../models/sla_model.pkl']

In [65]:
y = df["Resolution_Time_Hours"]

In [66]:
df["Resolution_Time_Hours"].describe()

count    1693.000000
mean       95.037803
std        55.852154
min         1.000000
25%        49.000000
50%        95.000000
75%       136.000000
max       239.000000
Name: Resolution_Time_Hours, dtype: float64

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [68]:
from sklearn.ensemble import RandomForestRegressor

resolution_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    random_state=42
)

In [69]:
resolution_model.fit(
    X_train,
    y_train
)

ValueError: Input y contains NaN.

In [70]:
df["Resolution_Time_Hours"].isnull().sum()

np.int64(307)

In [72]:
df["Resolution_Time_Hours"] = pd.to_numeric(
    df["Resolution_Time_Hours"],
    errors="coerce"
)

In [73]:
df["Resolution_Time_Hours"].isnull().sum()

np.int64(307)

In [74]:
df = df.dropna(
    subset=["Resolution_Time_Hours"]
)

In [75]:
df["Resolution_Time_Hours"].isnull().sum()

np.int64(0)

In [76]:
X = df["combined_text"]

y = df["Resolution_Time_Hours"]

In [77]:
X_tfidf = tfidf.fit_transform(X)

In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [79]:
resolution_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",25
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of 

In [80]:
preds = resolution_model.predict(
    X_test
)

In [81]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

mae = mean_absolute_error(
    y_test,
    preds
)

r2 = r2_score(
    y_test,
    preds
)

print("MAE:", mae)

print("R2:", r2)

MAE: 45.56512632629707
R2: 0.024105790148963635


In [82]:
joblib.dump(
    resolution_model,
    "../models/resolution_time_model.pkl"
)

['../models/resolution_time_model.pkl']